# Calibration Negatives - Improving Inter-Rater Kappa

## Problem
validate_synthetic_ollama.ipynb reported Fleiss' κ ≈ 0.00–0.09 ("Poor/Slight") despite 99.3% validity.
This is the **prevalence paradox**: when ~99% of items pass, kappa collapses regardless of actual judge consistency.

## Solution
Inject 15 **known-bad** synthetic reviews (3 per file) with deliberate flaws:
- Wrong sentiment (e.g. positive review labelled negative)
- Wrong aspect (e.g. review discusses price but labelled as packing)
- Unnatural text (spam/incoherent repetition)

## Methodology
1. Load existing 150 validated samples from `validation_results.csv`
2. Run judges **only** on the 15 calibration samples
3. Merge datasets → 165 total
4. Report kappa on **full set** (165), validity % on **originals only** (150)
5. Verify calibration detection rate: judges should reliably flag the bad samples

In [ ]:
print(" Starting: Loading imports and configuration...")
import time
_start_time = time.time()

import json
import random
import re
from datetime import datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd
import requests
from tqdm.auto import tqdm

# ─── Ollama Settings ──────────────────────────────────────────────
OLLAMA_BASE = "http://localhost:11434"

# Multiple judge models — each model evaluates every review independently
# to avoid inheriting the bias of a single model.
JUDGE_MODELS = [
    "llama3.1",   # Meta Llama 3.1 8B
    "mistral",    # Mistral 7B
    "gemma2",     # Google Gemma 2 9B
]

# ─── Runtime Config ──────────────────────────────────────────────────────
DATA_DIR   = Path("../data/augmented")
OUTPUT_DIR = Path("../data/augmented/validation_output")
# SAMPLE_N is not used in this notebook (calibration only, no re-sampling)
# Note: Only run the 15 calibration negatives through the judge pipeline

print(f"  Data dir    : {DATA_DIR.resolve()}")
print(f"  Output dir  : {OUTPUT_DIR.resolve()}")
print(f"  Judge models: {JUDGE_MODELS}")
print(f"  Total judges: {len(JUDGE_MODELS)} models x 3 personas = {len(JUDGE_MODELS)*3} per review")

print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Completed: Loading imports and configuration.")

 Starting: Loading imports and configuration...
  Data dir    : C:\Users\lucif\Desktop\Clearview\ml-research\data\augmented
  Output dir  : C:\Users\lucif\Desktop\Clearview\ml-research\data\augmented\validation_output
  Judge models: ['llama3.1', 'mistral', 'gemma2']
  Total judges: 3 models x 3 personas = 9 per review

 Done in 0.72s
 Completed: Loading imports and configuration.


In [2]:
print(" Starting: Setting domain configuration...")
_start_time = time.time()

SYNTHETIC_FILES = {
    "LLM_Gen_Packing_Neg_Reviews.csv":  ("packing",  "negative"),
    "LLM_Gen_Packing_Neu_Reviews.csv":  ("packing",  "neutral"),
    "LLM_Gen_Price_Neg_Reviews.csv":    ("price",    "negative"),
    "LLM_Gen_Price_Neu_Reviews.csv":    ("price",    "neutral"),
    "LLM_Gen_Smell_Neu_Reviews.csv":    ("smell",    "neutral"),
}

ASPECT_DESCRIPTIONS = {
    "packing":      "packaging, box, bottle, container, seal, wrapper",
    "price":        "cost, price, value for money, affordability",
    "smell":        "scent, fragrance, odour, aroma of the product",
    "texture":      "feel, consistency, thickness, spreadability on skin",
    "colour":       "colour, shade, pigmentation, hue",
    "shipping":     "delivery, courier, arrival condition",
    "stayingpower": "how long the product lasts, wear time, longevity",
}

SENTIMENT_DESCRIPTIONS = {
    "positive": "satisfaction, praise, or approval",
    "neutral":  "factual, balanced, or neither clearly positive nor negative",
    "negative": "dissatisfaction, complaint, or criticism",
}

CRITERIA = ["label_correctness", "aspect_relevance", "naturalness"]

print(f"  Files to validate : {len(SYNTHETIC_FILES)}")
print(f"  Aspects covered   : {sorted(set(v[0] for v in SYNTHETIC_FILES.values()))}")
print(f"  Criteria          : {CRITERIA}")

print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Completed: Setting domain configuration.")

 Starting: Setting domain configuration...
  Files to validate : 5
  Aspects covered   : ['packing', 'price', 'smell']
  Criteria          : ['label_correctness', 'aspect_relevance', 'naturalness']

 Done in 0.00s
 Completed: Setting domain configuration.


## Calibration Negatives

Each calibration sample has a **deliberate flaw** that a well-functioning judge should detect:

| ID | File | Flaw type | Description |
|---|---|---|---|
| calib_001 | Packing Neg | Wrong sentiment | Positive text labelled negative |
| calib_002 | Packing Neg | Wrong aspect | Discusses price, not packing |
| calib_003 | Packing Neg | Unnatural/spam | Keyword stuffing |
| calib_004 | Packing Neu | Wrong sentiment | Strongly negative text labelled neutral |
| calib_005 | Packing Neu | Wrong aspect | Discusses smell, not packing |
| calib_006 | Packing Neu | Unnatural/incoherent | Fragmented nonsense |
| calib_007 | Price Neg | Wrong sentiment | Positive text labelled negative |
| calib_008 | Price Neg | Wrong aspect | Discusses texture, not price |
| calib_009 | Price Neg | Unnatural/spam | Keyword stuffing |
| calib_010 | Price Neu | Wrong sentiment | Strongly positive text labelled neutral |
| calib_011 | Price Neu | Wrong aspect | Discusses colour, not price |
| calib_012 | Price Neu | Unnatural/incoherent | Fragmented nonsense |
| calib_013 | Smell Neu | Wrong sentiment | Strongly positive text labelled neutral |
| calib_014 | Smell Neu | Wrong aspect | Discusses shipping, not smell |
| calib_015 | Smell Neu | Unnatural/spam | Keyword stuffing |

In [3]:
CALIBRATION_NEGATIVES = [
    # ─── LLM_Gen_Packing_Neg_Reviews.csv (aspect=packing, sentiment=negative) ───
    {
        "calib_id": "calib_001",
        "file": "LLM_Gen_Packing_Neg_Reviews.csv",
        "aspect": "packing",
        "sentiment": "negative",
        "review_text": "The packaging was absolutely stunning! Beautifully wrapped and arrived in perfect condition. I loved how it was presented.",
        "flaw": "Wrong sentiment (positive expressed, not negative)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_002",
        "file": "LLM_Gen_Packing_Neg_Reviews.csv",
        "aspect": "packing",
        "sentiment": "negative",
        "review_text": "The price was outrageously expensive for what you get. Not worth the money at all.",
        "flaw": "Wrong aspect (discusses price, not packing)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_003",
        "file": "LLM_Gen_Packing_Neg_Reviews.csv",
        "aspect": "packing",
        "sentiment": "negative",
        "review_text": "packaging packaging BAD packaging terrible packaging packaging damaged packaging packaging awful box box box",
        "flaw": "Unnatural/spam (keyword stuffing)",
        "expected_valid": False,
    },
    # ─── LLM_Gen_Packing_Neu_Reviews.csv (aspect=packing, sentiment=neutral) ───
    {
        "calib_id": "calib_004",
        "file": "LLM_Gen_Packing_Neu_Reviews.csv",
        "aspect": "packing",
        "sentiment": "neutral",
        "review_text": "The packaging was absolutely destroyed and disgusting! Worst I have ever seen! Complete disaster terrible!",
        "flaw": "Wrong sentiment (strongly negative, not neutral)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_005",
        "file": "LLM_Gen_Packing_Neu_Reviews.csv",
        "aspect": "packing",
        "sentiment": "neutral",
        "review_text": "The scent is very floral and quite strong. It lingers for hours after application on the skin.",
        "flaw": "Wrong aspect (discusses smell, not packing)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_006",
        "file": "LLM_Gen_Packing_Neu_Reviews.csv",
        "aspect": "packing",
        "sentiment": "neutral",
        "review_text": "Packing was. The box arrived. Neither bad nor good. Packing. Neutral feelings about this thing exist maybe.",
        "flaw": "Unnatural/incoherent text",
        "expected_valid": False,
    },
    # ─── LLM_Gen_Price_Neg_Reviews.csv (aspect=price, sentiment=negative) ───
    {
        "calib_id": "calib_007",
        "file": "LLM_Gen_Price_Neg_Reviews.csv",
        "aspect": "price",
        "sentiment": "negative",
        "review_text": "Great value for money! The price is very reasonable and totally worth every cent. Highly recommended!",
        "flaw": "Wrong sentiment (positive expressed, not negative)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_008",
        "file": "LLM_Gen_Price_Neg_Reviews.csv",
        "aspect": "price",
        "sentiment": "negative",
        "review_text": "The texture is incredibly smooth and blends easily on the skin. Very silky and comfortable to wear.",
        "flaw": "Wrong aspect (discusses texture, not price)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_009",
        "file": "LLM_Gen_Price_Neg_Reviews.csv",
        "aspect": "price",
        "sentiment": "negative",
        "review_text": "price expensive very much too much price money not good price bad money expensive price cost too much price",
        "flaw": "Unnatural/spam (keyword stuffing)",
        "expected_valid": False,
    },
    # ─── LLM_Gen_Price_Neu_Reviews.csv (aspect=price, sentiment=neutral) ───
    {
        "calib_id": "calib_010",
        "file": "LLM_Gen_Price_Neu_Reviews.csv",
        "aspect": "price",
        "sentiment": "neutral",
        "review_text": "Absolutely amazing price! The best deal I have ever found in my life! So incredibly cheap and wonderful!",
        "flaw": "Wrong sentiment (strongly positive, not neutral)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_011",
        "file": "LLM_Gen_Price_Neu_Reviews.csv",
        "aspect": "price",
        "sentiment": "neutral",
        "review_text": "The colour is a beautiful deep red, very vibrant and pigmented. Great colour payoff on the lips.",
        "flaw": "Wrong aspect (discusses colour, not price)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_012",
        "file": "LLM_Gen_Price_Neu_Reviews.csv",
        "aspect": "price",
        "sentiment": "neutral",
        "review_text": "Price. Not cheap not expensive. Price range is within price. The price of this product is the priced.",
        "flaw": "Unnatural/incoherent text",
        "expected_valid": False,
    },
    # ─── LLM_Gen_Smell_Neu_Reviews.csv (aspect=smell, sentiment=neutral) ───
    {
        "calib_id": "calib_013",
        "file": "LLM_Gen_Smell_Neu_Reviews.csv",
        "aspect": "smell",
        "sentiment": "neutral",
        "review_text": "The smell is absolutely divine and I am completely obsessed with this incredible fragrance! Best scent ever!",
        "flaw": "Wrong sentiment (strongly positive, not neutral)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_014",
        "file": "LLM_Gen_Smell_Neu_Reviews.csv",
        "aspect": "smell",
        "sentiment": "neutral",
        "review_text": "The delivery was fast and took only two days. The package came sealed and in good condition.",
        "flaw": "Wrong aspect (discusses shipping, not smell)",
        "expected_valid": False,
    },
    {
        "calib_id": "calib_015",
        "file": "LLM_Gen_Smell_Neu_Reviews.csv",
        "aspect": "smell",
        "sentiment": "neutral",
        "review_text": "smell scent neutral smell not bad not good smell scent fragrance neither positive negative smell odor smell",
        "flaw": "Unnatural/spam (keyword stuffing)",
        "expected_valid": False,
    },
]

print(f"  Calibration negatives defined: {len(CALIBRATION_NEGATIVES)}")
flaw_types = {}
for s in CALIBRATION_NEGATIVES:
    ft = s['flaw'].split('(')[0].strip()
    flaw_types[ft] = flaw_types.get(ft, 0) + 1
for ft, count in flaw_types.items():
    print(f"    {ft}: {count}")

  Calibration negatives defined: 15
    Wrong sentiment: 5
    Wrong aspect: 5
    Unnatural/spam: 3
    Unnatural/incoherent text: 2


## Judge Setup

Three personas provide diversity:
- **Judge_A** (strict): Marks FAIL on ambiguity - highest recall for flawed samples
- **Judge_B** (moderate): Balanced standards - baseline assessor
- **Judge_C** (lenient): Gives benefit of the doubt - tests whether obvious flaws still get caught

In [4]:
print(" Starting: Defining judge personas and prompt template...")
_start_time = time.time()

JUDGE_PERSONAS = [
    {
        "name": "Judge_A",
        "system": (
            "You are a strict NLP annotation expert specialising in Aspect-Based Sentiment Analysis. "
            "You evaluate synthetic cosmetic reviews with high standards. "
            "If a review is ambiguous, mark it FAIL. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
    {
        "name": "Judge_B",
        "system": (
            "You are a senior data quality assessor for an NLP research team. "
            "Validate whether machine-generated cosmetic reviews are suitable for training a sentiment classifier. "
            "Apply consistent, moderate standards. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
    {
        "name": "Judge_C",
        "system": (
            "You are a consumer research analyst experienced with cosmetic product reviews. "
            "Assess whether auto-generated reviews are realistic and correctly labelled. "
            "Give benefit of the doubt when intent is clear. "
            "You MUST respond ONLY with a valid JSON object. No explanation, no extra text."
        ),
    },
]

JUDGE_PROMPT = """Evaluate this synthetic cosmetic product review.

REVIEW TEXT:
\"\"\"{review_text}\"\"\"

CLAIMED ASPECT: {aspect}
(Review should discuss: {aspect_description})

CLAIMED SENTIMENT: {sentiment}
(Review should express: {sentiment_description})

Evaluate on exactly three criteria. Respond ONLY with this JSON, nothing else:

{{
  "label_correctness": {{"verdict": "PASS", "reason": "one sentence why"}},
  "aspect_relevance":  {{"verdict": "PASS", "reason": "one sentence why"}},
  "naturalness":       {{"verdict": "PASS", "reason": "one sentence why"}}
}}

Rules:
- label_correctness = PASS if review clearly expresses {sentiment} sentiment toward {aspect}
- aspect_relevance  = PASS if review actually discusses {aspect} content
- naturalness       = PASS if review reads like a genuine human-written cosmetic review
- Use FAIL if a criterion is not met
- Output ONLY the JSON object, no other text before or after
"""

print(f"  Judges defined : {[j['name'] for j in JUDGE_PERSONAS]}")

print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Completed: Defining judge personas and prompt template.")

 Starting: Defining judge personas and prompt template...
  Judges defined : ['Judge_A', 'Judge_B', 'Judge_C']

 Done in 0.00s
 Completed: Defining judge personas and prompt template.


In [5]:
print(" Starting: Defining Ollama API functions...")
_start_time = time.time()
import time as t_lib

def check_ollama(model: str) -> str:
    """Check Ollama is running, model exists, and determine hardware (GPU/CPU).
    Returns the resolved full model tag (e.g. 'llama3.1:latest')."""
    try:
        r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
        models = [m["name"] for m in r.json().get("models", [])]
    except requests.exceptions.ConnectionError:
        print("\n[ERROR] Cannot connect to Ollama at http://localhost:11434")
        raise RuntimeError("Ollama not reachable")

    matched = [m for m in models if model.split(":")[0] in m]
    if not matched:
        print(f"\n[ERROR] Model '{model}' not found locally. Run: ollama pull {model}")
        raise RuntimeError(f"Model '{model}' not available")

    hw_info = "[Detecting...]"
    for attempt in range(3):
        try:
            requests.post(
                f"{OLLAMA_BASE}/api/generate",
                json={"model": matched[0], "prompt": "hi", "stream": False},
                timeout=10,
            )
            r_ps = requests.get(f"{OLLAMA_BASE}/api/ps", timeout=5)
            ps_data = r_ps.json().get("models", [])
            model_ps = next((m for m in ps_data if matched[0] in m["name"]), None)
            if model_ps:
                proc = model_ps.get("processor", "Unknown")
                if proc == "Unknown":
                    t_lib.sleep(2)
                    continue
                hw_info = f"Running on {proc}"
                break
            else:
                hw_info = "Hardware: Ollama will choose best available"
                t_lib.sleep(1)
        except Exception:
            hw_info = "Hardware: Native (check nvidia-smi for VRAM usage)"

    print(f"  \u2705 {matched[0]:30s} \u2014 {hw_info}")
    return matched[0]


def check_all_models(judge_models: list) -> dict:
    """Check all judge models are available. Returns {model_name: resolved_tag}."""
    print("  Checking judge models:")
    resolved = {}
    for m in judge_models:
        tag = check_ollama(m)
        resolved[m] = tag
    return resolved


def call_ollama(system_prompt: str, user_prompt: str, model: str) -> str:
    """Send a chat request to local Ollama and return the response text."""
    payload = {
        "model":  model,
        "stream": False,
        "options": {"temperature": 0.1, "top_p": 0.9},
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
    }
    response = requests.post(f"{OLLAMA_BASE}/api/chat", json=payload, timeout=120)
    response.raise_for_status()
    return response.json()["message"]["content"]


def call_judge(judge: dict, review_text: str, aspect: str,
               sentiment: str, model: str) -> dict | None:
    """Call one judge persona on a specific model. Retries 3x."""
    prompt = JUDGE_PROMPT.format(
        review_text=review_text[:500],
        aspect=aspect,
        aspect_description=ASPECT_DESCRIPTIONS.get(aspect, aspect),
        sentiment=sentiment,
        sentiment_description=SENTIMENT_DESCRIPTIONS.get(sentiment, sentiment),
    )
    for attempt in range(3):
        try:
            raw = call_ollama(judge["system"], prompt, model).strip()
            raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
            raw = re.sub(r"\s*```$",          "", raw, flags=re.MULTILINE)
            match = re.search(r"\{.*\}", raw, re.DOTALL)
            if match:
                raw = match.group(0)
            return json.loads(raw)
        except json.JSONDecodeError:
            t_lib.sleep(1)
        except Exception:
            t_lib.sleep(2)
    return None


print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Completed: Defining Ollama API functions.")

 Starting: Defining Ollama API functions...

 Done in 0.00s
 Completed: Defining Ollama API functions.


In [6]:
print(" Starting: Defining Fleiss' Kappa functions...")
_start_time = time.time()

def fleiss_kappa(ratings_matrix: np.ndarray) -> float:
    """
    Fleiss' Kappa for multi-rater binary agreement.
    """
    n_items, n_categories = ratings_matrix.shape
    n_raters = int(ratings_matrix[0].sum())
    if n_raters < 2:
        return float("nan")

    p_j   = ratings_matrix.sum(axis=0) / (n_items * n_raters)
    P_e   = (p_j ** 2).sum()
    P_i   = ((ratings_matrix ** 2).sum(axis=1) - n_raters) / (n_raters * (n_raters - 1))
    P_bar = P_i.mean()

    if abs(1 - P_e) < 1e-9:
        return 1.0
    return float((P_bar - P_e) / (1 - P_e))


def interpret_kappa(k: float) -> str:
    """Return a human-readable label for a Fleiss' Kappa value."""
    if np.isnan(k): return "N/A"
    if k < 0:       return "Poor"
    if k < 0.20:    return "Slight"
    if k < 0.40:    return "Fair"
    if k < 0.60:    return "Moderate"
    if k < 0.80:    return "Substantial"
    return "Almost Perfect"


print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Completed: Defining Fleiss' Kappa functions.")

 Starting: Defining Fleiss' Kappa functions...

 Done in 0.00s
 Completed: Defining Fleiss' Kappa functions.


## Step 1 - Load Existing Results

Load the 150 validated samples from 04_validate_synthetic_ollama.ipynb's output CSV.
We will **not** re-run these through the judge pipeline - only the 15 new calibration samples will be evaluated.

In [7]:
existing_csv = OUTPUT_DIR / "validation_results.csv"
df_existing = pd.read_csv(existing_csv)
print(f"Loaded {len(df_existing)} existing results from {existing_csv}")
print(f"Original validity: {df_existing['overall_valid'].sum()}/{len(df_existing)} "
      f"({100*df_existing['overall_valid'].mean():.1f}%)")
print(f"\nColumns ({len(df_existing.columns)}):")
for col in df_existing.columns:
    print(f"  {col}")

Loaded 150 existing results from ..\data\augmented\validation_output\validation_results.csv
Original validity: 149/150 (99.3%)

Columns (75):
  sample_id
  file
  aspect
  sentiment
  review_text
  llama3.1_Judge_A_label_correctness
  llama3.1_Judge_A_label_correctness_reason
  llama3.1_Judge_A_aspect_relevance
  llama3.1_Judge_A_aspect_relevance_reason
  llama3.1_Judge_A_naturalness
  llama3.1_Judge_A_naturalness_reason
  llama3.1_Judge_B_label_correctness
  llama3.1_Judge_B_label_correctness_reason
  llama3.1_Judge_B_aspect_relevance
  llama3.1_Judge_B_aspect_relevance_reason
  llama3.1_Judge_B_naturalness
  llama3.1_Judge_B_naturalness_reason
  llama3.1_Judge_C_label_correctness
  llama3.1_Judge_C_label_correctness_reason
  llama3.1_Judge_C_aspect_relevance
  llama3.1_Judge_C_aspect_relevance_reason
  llama3.1_Judge_C_naturalness
  llama3.1_Judge_C_naturalness_reason
  llama3.1_majority_label_correctness
  llama3.1_majority_aspect_relevance
  llama3.1_majority_naturalness
  mistral_

## Step 2 - Run Judges on Calibration Negatives

Run the exact same judge pipeline from notebook 22 on the 15 calibration samples only.
Each row will have identical column structure to `df_existing` plus extra calibration columns.

In [ ]:
print(" Starting: Checking models and preparing calibration run...")
_start_time = time.time()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Resolve all judge model tags (fails fast if any model is not pulled)
RESOLVED_MODELS = check_all_models(JUDGE_MODELS)
print(f"\n  Resolved models: {RESOLVED_MODELS}")

model_keys = [m.split(":")[0] for m in RESOLVED_MODELS.keys()]
n_calib = len(CALIBRATION_NEGATIVES)
total_calib_calls = n_calib * len(JUDGE_MODELS) * len(JUDGE_PERSONAS)
print(f"\n  Calibration samples : {n_calib}")
print(f"  Judge calls expected: {total_calib_calls}  ({n_calib} x {len(JUDGE_MODELS)} models x {len(JUDGE_PERSONAS)} personas)")
print(f"\n Done in {time.time() - _start_time:.2f}s")
print(" Model check complete.")

# ───────────────────────────────────────────────────────────────────────
print(f"\n Running calibration evaluation ({n_calib} samples x {len(JUDGE_MODELS)} models x {len(JUDGE_PERSONAS)} personas)...\n")
_start_eval_time = time.time()
calib_results = []

pbar = tqdm(CALIBRATION_NEGATIVES, desc="Calibration Samples", unit="sample")
for i, sample in enumerate(pbar):
    print(f"\nCalib [{i+1}/{n_calib}] | {sample['calib_id']} | {sample['aspect']:10s} | "
          f"{sample['sentiment']:8s} | {sample['review_text'][:50]}...")
    print(f"  Flaw: {sample['flaw']}")

    row = {
        # Use a high sample_id to avoid collision with existing 1-150
        "sample_id":   1000 + i + 1,
        "file":        sample["file"],
        "aspect":      sample["aspect"],
        "sentiment":   sample["sentiment"],
        "review_text": sample["review_text"],
        # Calibration-specific columns
        "calib_id":       sample["calib_id"],
        "is_calibration": True,
        "calib_flaw":     sample["flaw"],
        "expected_valid": 0,
    }

    # model_short_name -> {criterion -> list of persona scores}
    model_persona_scores: dict[str, dict[str, list[int]]] = {}

    for model_name, model_tag in RESOLVED_MODELS.items():
        model_key = model_name.split(":")[0]   # e.g. "llama3.1"
        model_persona_scores[model_key] = {c: [] for c in CRITERIA}

        print(f"  \u250c\u2500 Model: {model_key}")
        for judge in JUDGE_PERSONAS:
            verdict = call_judge(judge, sample["review_text"],
                                 sample["aspect"], sample["sentiment"], model_tag)

            col_prefix = f"{model_key}_{judge['name']}"
            if verdict is None:
                for c in CRITERIA:
                    row[f"{col_prefix}_{c}"] = "ERROR"
                    row[f"{col_prefix}_{c}_reason"] = ""
                print(f"  \u2502  \u2514\u2500 {judge['name']:7s}: [ERROR \u2014 all criteria]")
                continue

            parsed = {}
            for c in CRITERIA:
                v = verdict.get(c, {}).get("verdict", "FAIL")
                score = 1 if str(v).upper() == "PASS" else 0
                parsed[c] = score
                model_persona_scores[model_key][c].append(score)
                row[f"{col_prefix}_{c}"] = str(v).upper()
                row[f"{col_prefix}_{c}_reason"] = verdict.get(c, {}).get("reason", "")

            _TICK, _CROSS = "✓", "✗"
            icons = " ".join(f"[{_TICK if parsed[c] else _CROSS} {c[0].upper()}]" for c in CRITERIA)
            print(f"  \u2502  \u2514\u2500 {judge['name']:7s}: {icons}")

        # Per-model majority vote (>= 2/3 personas)
        for c in CRITERIA:
            votes = model_persona_scores[model_key][c]
            majority = (1 if sum(votes) >= 2 else 0) if len(votes) >= 2 else 0
            row[f"{model_key}_majority_{c}"] = majority
        _TICK2, _CROSS2 = "✓", "✗"
        model_summary = "  ".join(
            f"{c[0].upper()}={_TICK2 if row[f'{model_key}_majority_{c}'] else _CROSS2}"
            for c in CRITERIA
        )
        print(f"  \u2514\u2500 {model_key} majority: {model_summary}")

    # Cross-model final verdict: criterion passes if >= ceil(n_models/2)+1 models agree
    n_models = len(RESOLVED_MODELS)
    pass_threshold = (n_models // 2) + 1   # e.g. 2 out of 3
    for c in CRITERIA:
        model_votes = [row.get(f"{mk}_majority_{c}", 0) for mk in model_persona_scores]
        row[f"final_{c}"] = 1 if sum(model_votes) >= pass_threshold else 0
        row[f"avg_score_{c}"] = round(sum(model_votes) / n_models, 3) if n_models else 0

    row["overall_valid"] = int(all(row.get(f"final_{c}", 0) == 1 for c in CRITERIA))

    expected = "FAIL" if sample["expected_valid"] == False else "PASS"
    actual   = "PASS" if row["overall_valid"] == 1 else "FAIL"
    caught   = "[DETECTED]" if actual == "FAIL" else "[MISSED ⚠]"
    print(f"  Result: expected={expected}  actual={actual}  {caught}")

    calib_results.append(row)

total_calib_duration = time.time() - _start_eval_time
print(f"\n Total Calibration Evaluation Time: {timedelta(seconds=int(total_calib_duration))}")
print(" Completed: Calibration judge evaluation.")

 Starting: Checking models and preparing calibration run...
  Checking judge models:


## Step 3 - Merge and Compute Statistics

Combine the 150 original results with the 15 calibration results to form a 165-sample dataset.
- **Validity %** is reported on originals only (150)
- **Fleiss' Kappa** is computed on the full combined set (165)

In [ ]:
# Add calibration markers to existing df
df_existing["is_calibration"] = False
df_existing["expected_valid"] = 1
df_existing["calib_flaw"] = ""
df_existing["calib_id"] = ""

# Build calib results df
df_calib = pd.DataFrame(calib_results)

# Merge — align columns (fill missing with NaN)
df_combined = pd.concat([df_existing, df_calib], ignore_index=True)

print(f"Combined dataset: {len(df_combined)} rows")
print(f"  Original samples    : {len(df_existing)}")
print(f"  Calibration samples : {len(df_calib)}")
print(f"  Total columns       : {len(df_combined.columns)}")

# ─── Validity on originals only ───
df_orig = df_combined[~df_combined["is_calibration"]]
n_orig_valid = int(df_orig["overall_valid"].sum())
pct_orig = 100 * n_orig_valid / len(df_orig)
print(f"\nOriginal validity: {n_orig_valid}/{len(df_orig)} ({pct_orig:.1f}%)")

# ─── Calibration detection rate ───
df_calib_only = df_combined[df_combined["is_calibration"]]
n_detected = int((df_calib_only["overall_valid"] == 0).sum())
detection_rate = 100 * n_detected / len(df_calib_only)
print(f"\nCalibration detection: {n_detected}/{len(df_calib_only)} ({detection_rate:.1f}%)")
print(f"  (Number of known-bad samples correctly flagged as FAIL)")

# ─── Fleiss' Kappa on combined set (165) ───
# Treat every model-persona combination as an independent rater
all_judge_cols = [
    f"{mk}_{j['name']}"
    for mk in model_keys
    for j in JUDGE_PERSONAS
]

kappas_combined = {}
for c in CRITERIA:
    ratings = []
    for _, r in df_combined.iterrows():
        passes = sum(
            1 for col in all_judge_cols
            if str(r.get(f"{col}_{c}", "FAIL")).upper() == "PASS"
        )
        n_raters = len(all_judge_cols)
        ratings.append([passes, n_raters - passes])
    kappas_combined[c] = fleiss_kappa(np.array(ratings))

print("\nKappa (combined set, 165 samples):")
for c, k in kappas_combined.items():
    print(f"  {c:25s}: kappa={k:.3f}  ({interpret_kappa(k)})")

# ─── Baseline kappa from notebook 22 for comparison ───
# Recompute on original 150 only
kappas_baseline = {}
for c in CRITERIA:
    ratings = []
    for _, r in df_orig.iterrows():
        passes = sum(
            1 for col in all_judge_cols
            if str(r.get(f"{col}_{c}", "FAIL")).upper() == "PASS"
        )
        n_raters = len(all_judge_cols)
        ratings.append([passes, n_raters - passes])
    kappas_baseline[c] = fleiss_kappa(np.array(ratings))

print("\nKappa (originals only, 150 samples = baseline):")
for c, k in kappas_baseline.items():
    print(f"  {c:25s}: kappa={k:.3f}  ({interpret_kappa(k)})")

## Step 4 - Results Summary

In [ ]:
sep = "=" * 70
print(sep)
print("CALIBRATION-AUGMENTED VALIDATION RESULTS")
print(sep)
print(f"  Total samples     : {len(df_combined)}  (150 original + 15 calibration)")
print(f"  Original validity : {n_orig_valid}/{len(df_orig)} ({pct_orig:.1f}%)  <- unchanged from notebook 22")
print(f"  Calib detected    : {n_detected}/{len(df_calib_only)} ({detection_rate:.1f}%)  <- judges correctly flagged bad samples")
print()
print("  \u2500\u2500 Kappa (combined set, 165 samples) " + "-" * 36)
for c, k in kappas_combined.items():
    label = c.replace("_", " ").title()
    print(f"  {label:22s} : kappa={k:+.3f}  ({interpret_kappa(k)})")
print()
print("  \u2500\u2500 Comparison to baseline (150-sample kappa) " + "-" * 28)
print(f"  {'Criterion':<22}   {'Before (k)':>10}   {'After (k)':>10}   {'Change':>10}")
print("  " + "-" * 62)
for c in CRITERIA:
    label  = c.replace("_", " ").title()
    before = kappas_baseline[c]
    after  = kappas_combined[c]
    change = after - before
    sign   = "+" if change >= 0 else ""
    print(f"  {label:<22}   {before:>+10.3f}   {after:>+10.3f}   {sign}{change:>9.3f}")
print()
print("  \u2500\u2500 Calibration breakdown by flaw type " + "-" * 34)

# Per-flaw-type detection
flaw_groups = {}
for _, r in df_calib_only.iterrows():
    ft = str(r["calib_flaw"]).split("(")[0].strip()
    if ft not in flaw_groups:
        flaw_groups[ft] = {"total": 0, "detected": 0}
    flaw_groups[ft]["total"] += 1
    if r["overall_valid"] == 0:
        flaw_groups[ft]["detected"] += 1

for ft, stats in flaw_groups.items():
    pct_ft = 100 * stats["detected"] / stats["total"]
    print(f"  {ft:<35} : {stats['detected']}/{stats['total']} ({pct_ft:.0f}%)")

print(sep)

In [ ]:
# Save combined results
calibrated_csv = OUTPUT_DIR / "validation_results_calibrated.csv"
df_combined.to_csv(calibrated_csv, index=False)
print(f"Saved combined results ({len(df_combined)} rows) to:")
print(f"  {calibrated_csv.resolve()}")
print(f"\nFile size: {calibrated_csv.stat().st_size / 1024:.1f} KB")

In [ ]:
# Display calibration sample results table
display_cols = [
    "calib_id", "aspect", "sentiment", "calib_flaw",
    "overall_valid", "final_label_correctness", "final_aspect_relevance", "final_naturalness",
]
df_calib_display = df_calib_only[display_cols].copy()
df_calib_display["detected"] = df_calib_display["overall_valid"].apply(
    lambda v: "YES (flagged FAIL)" if v == 0 else "NO (missed)"
)

print("Calibration sample detection results:")
print(f"{'calib_id':<12} {'aspect':<10} {'sentiment':<10} {'detected':<20} {'flaw'}")
print("-" * 90)
for _, row in df_calib_display.iterrows():
    print(f"{row['calib_id']:<12} {row['aspect']:<10} {row['sentiment']:<10} "
          f"{row['detected']:<20} {row['calib_flaw']}")

print("\n")

try:
    from IPython.display import display
    display(df_calib_only[display_cols])
except Exception:
    print(df_calib_only[display_cols].to_string())